In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn import svm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from itertools import product

from src.models.train_model import train_model

e:\miniconda\envs\mlops\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
project_dir = Path(Path.cwd()).parents[0]
train_set = pd.read_csv(
    project_dir / "data" / "processed" / "dataset_train.csv",
    index_col="Unnamed: 0",
)
valid_set = pd.read_csv(
    project_dir / "data" / "processed" / "dataset_val.csv",
    index_col="Unnamed: 0",
)

normalize_features = [
    i
    for i in train_set.columns
    if i
    not in [
        "_target",
        "Gender",
        "OverTime",
        "BusinessTravel",
        "Department",
        "EducationField",
        "JobRole",
        "MaritalStatus",
    ]
]

In [5]:
train_set.describe()

,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,TenureRatio,RoleEngagementScore,RoleStabilityRatio,ManagerStabilityRatio,PromotionRate,IncomePerYear,SalaryHikeExpectation,Gender_Male,OverTime_Yes,_target
count,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,...,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000,882.000000
mean,36.887755,0.393424,808.821995,0.719955,9.337868,2.899093,1.503401,2.705215,64.970522,2.748299,...,0.671067,5.738095,0.416974,1.035577,0.495200,693.464150,12.026077,0.595238,0.289116,0.160998
std,9.191609,0.675904,398.608506,0.523921,8.146594,1.034051,1.443798,1.081973,20.065026,0.710008,...,0.329495,3.441171,0.295505,0.725628,0.581925,442.719101,3.368816,0.491124,0.453609,0.367737
min,18.000000,0.000000,103.000000,0.000000,1.000000,1.000000,0.000000,1.000000,30.000000,1.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000
25%,30.000000,0.000000,479.250000,0.000000,2.000000,2.000000,0.000000,2.000000,48.000000,2.000000,...,0.385684,3.000000,0.174552,0.800000,0.200000,423.937500,9.000000,0.000000,0.000000,0.000000
50%,35.000000,0.000000,801.500000,1.000000,7.000000,3.000000,2.000000,3.000000,65.000000,3.000000,...,0.785714,6.000000,0.400000,1.000000,0.300000,601.111111,11.000000,1.000000,0.000000,0.000000
75%,43.000000,1.000000,1157.750000,1.000000,14.000000,4.000000,2.000000,4.000000,82.000000,3.000000,...,1.000000,6.000000,0.666667,1.000000,0.500000,839.805556,15.000000,1.000000,1.000000,0.000000
max,60.000000,2.000000,1499.000000,2.000000,29.000000,5.000000,5.000000,4.000000,100.000000,4.000000,...,1.000000,20.000000,1.000000,9.000000,5.000000,2994.000000,21.000000,1.000000,1.000000,1.000000


In [3]:
print(normalize_features)

['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyRate', 'NumCompaniesWorked', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsSinceLastPromotion', 'TenureRatio', 'RoleEngagementScore', 'RoleStabilityRatio', 'ManagerStabilityRatio', 'PromotionRate', 'IncomePerYear', 'SalaryHikeExpectation', 'Gender_Male', 'OverTime_Yes']


In [ ]:
scaler = ColumnTransformer(
    transformers=[("columns scaler", StandardScaler(), normalize_features)],
    remainder="passthrough",
).set_output(transform="pandas")

In [3]:
C = 1.0  # SVM regularization parameter
models = (
    svm.SVC(kernel="linear", C=C),
    svm.LinearSVC(C=C, max_iter=10000),
    svm.SVC(kernel="rbf", gamma=0.7, C=C),
    svm.SVC(kernel="poly", degree=3, gamma="auto", C=C),
)

for clf in models:
    estimator = Pipeline([("scaler", scaler), ("clf", clf)])

    train_model(
        estimator,
        train_dataset=train_set,
        valid_dataset=valid_set,
        experiment_name="linear_models_",
    )

print("done")

2026/06/15 00:36:14 INFO mlflow.tracking.fluent: Experiment with name 'linear_models_' does not exist. Creating a new experiment.
2026/06/15 00:36:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Оптимальный порог по f1-score: 1.0000
F1 Score: 0.4375
ROC AUC: 0.6417682926829268
PR AUC: 0.3708545918367347
Confusion matrix: 
 [[244   2]
 [ 34  14]]


2026/06/15 00:36:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run thoughtful-deer-675 at: http://127.0.0.1:5000/#/experiments/1/runs/ef5913f6539a4147a581b03fd624f5e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/15 00:36:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run sincere-fox-952 at: http://127.0.0.1:5000/#/experiments/1/runs/14d82a20edb4426f8c623039927790c3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/15 00:36:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Оптимальный порог по f1-score: 1.0000
F1 Score: 0.36363636363636365
ROC AUC: 0.6128048780487805
PR AUC: 0.2891156462585034
Confusion matrix: 
 [[240   6]
 [ 36  12]]


2026/06/15 00:36:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gregarious-fowl-704 at: http://127.0.0.1:5000/#/experiments/1/runs/dae3244000ec4c1ab4fb03ad27d46e03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
done
